In [12]:
import pandas as pd
import pyodbc

In [13]:
# ============ STEP 1: LOAD EXCEL FILE ============
print("📥 Loading Excel file...")
excel_file = r'Online Retail.xlsx'  # ← Change path

df = pd.read_excel(excel_file, sheet_name='Online Retail')
print(f"✅ Loaded {len(df)} rows")

📥 Loading Excel file...
✅ Loaded 541909 rows


In [14]:
# ============ STEP 2: CLEAN & PREPARE DATA ============
print("\n🧹 STEP 2: Cleaning & preparing data...")
try:
    # Rename columns
    df.columns = [
        'InvoiceNo', 'StockCode', 'Description', 'Quantity', 
        'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country'
    ]
    
    # Remove invalid rows
    df = df.dropna(subset=['CustomerID', 'InvoiceDate', 'Quantity'])
    
    # Remove cancelled orders (negative quantities)
    df = df[df['Quantity'] > 0]
    df = df[df['UnitPrice'] > 0]
    
    # Calculate Revenue
    df['Revenue'] = df['Quantity'] * df['UnitPrice']
    
    # Select only needed columns
    df = df[['InvoiceNo', 'InvoiceDate', 'CustomerID', 'Quantity', 'UnitPrice', 'Country', 'Revenue']]
    
    print(f"✅ Cleaned: {len(df):,} rows ready for SQL")
except Exception as e:
    print(f"❌ Error cleaning data: {e}")
    exit()


🧹 STEP 2: Cleaning & preparing data...
✅ Cleaned: 397,884 rows ready for SQL


In [15]:
# ============ STEP 3: CONNECT TO SQL SERVER ============
print("\n🔌 STEP 3: Connecting to SQL Server...")
try:
    conn = pyodbc.connect(
        'Driver={ODBC Driver 17 for SQL Server};'
        'Server=(localdb)\\MSSQLLocalDB;'
        'Database=RevOpsAnalytics;'
        'Trusted_Connection=yes;'
    )
    cursor = conn.cursor()
    print("✅ Connected to SQL Server successfully!")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("\n   Troubleshooting:")
    print("   1. Run: SqlLocalDB start MSSQLLocalDB")
    print("   2. Verify: sqlcmd -S (localdb)\\MSSQLLocalDB -E")
    print("   3. Check: Database 'RevOpsAnalytics' exists")
    print("   4. Check: Table 'transactions' exists")
    exit()



🔌 STEP 3: Connecting to SQL Server...
✅ Connected to SQL Server successfully!


In [16]:
# ============ STEP 4: CLEAR EXISTING DATA ============
print("\n🗑️  STEP 4: Clearing existing data from table...")
try:
    cursor.execute("DELETE FROM transactions")
    conn.commit()
    print("✅ Table cleared")
except Exception as e:
    print(f"⚠️  Could not clear table: {e}")
    print("   (This is OK if table is empty)")


🗑️  STEP 4: Clearing existing data from table...
✅ Table cleared


In [17]:
# ============ STEP 5: INSERT DATA INTO SQL ============
print("\n📤 STEP 5: Inserting data into SQL Server...")
print("   (This may take 2-5 minutes for 400K+ rows)\n")

batch_size = 2000
total_rows = len(df)
inserted_count = 0
error_count = 0

try:
    for i in range(0, total_rows, batch_size):
        batch = df.iloc[i:i+batch_size]
        
        for idx, row in batch.iterrows():
            try:
                # FIXED: Don't insert TransactionID (let SQL auto-generate it)
                cursor.execute(
                    '''INSERT INTO transactions 
                       (InvoiceNo, InvoiceDate, CustomerID, Quantity, UnitPrice, Country, Revenue)
                       VALUES (?, ?, ?, ?, ?, ?, ?)''',
                    str(row['InvoiceNo']),
                    pd.to_datetime(row['InvoiceDate']),
                    int(row['CustomerID']),
                    int(row['Quantity']),
                    float(row['UnitPrice']),
                    str(row['Country']),
                    float(row['Revenue'])
                )
                inserted_count += 1
            except Exception as e:
                error_count += 1
                if error_count <= 3:
                    print(f"   ⚠️  Row {idx}: {str(e)[:60]}")
        
        conn.commit()
        
        progress = min(i + batch_size, total_rows)
        percentage = (progress / total_rows) * 100
        bar_length = 40
        filled = int(bar_length * progress / total_rows)
        bar = '█' * filled + '░' * (bar_length - filled)
        
        print(f"   [{bar}] {progress:,}/{total_rows:,} rows ({percentage:.1f}%)")
    
    print(f"\n✅ Data insertion complete!")
    print(f"   Inserted: {inserted_count:,} rows")
    if error_count > 0:
        print(f"   Errors: {error_count:,} rows (skipped)")

except Exception as e:
    print(f"❌ Error during insertion: {e}")
    conn.close()
    exit()



📤 STEP 5: Inserting data into SQL Server...
   (This may take 2-5 minutes for 400K+ rows)

   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 2,000/397,884 rows (0.5%)
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 4,000/397,884 rows (1.0%)
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 6,000/397,884 rows (1.5%)
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 8,000/397,884 rows (2.0%)
   [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 10,000/397,884 rows (2.5%)
   [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 12,000/397,884 rows (3.0%)
   [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 14,000/397,884 rows (3.5%)
   [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 16,000/397,884 rows (4.0%)
   [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 18,000/397,884 rows (4.5%)
   [██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 20,000/397,884 rows (5.0%)
   [██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 22,000/397,884 rows (5.5%)
   [██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 24,000/397,884 rows (6.0%)
   [██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░

In [20]:
# ============ STEP 6: VERIFY DATA ============
print("\n📊 STEP 6: Verifying data in SQL...")

try:
    cursor.execute("SELECT COUNT(*) FROM transactions")
    row_count = cursor.fetchone()[0]
    print(f"✅ Total rows in database: {row_count:,}")
    
    cursor.execute("SELECT SUM(Revenue) FROM transactions")
    result = cursor.fetchone()[0]
    total_revenue = result if result else 0
    print(f"✅ Total revenue: ${total_revenue:,.2f}")
    
    cursor.execute("SELECT COUNT(DISTINCT CustomerID) FROM transactions")
    customer_count = cursor.fetchone()[0]
    print(f"✅ Unique customers: {customer_count:,}")
    
    cursor.execute("SELECT COUNT(DISTINCT Country) FROM transactions")
    country_count = cursor.fetchone()[0]
    print(f"✅ Countries: {country_count}")
    
    cursor.execute("SELECT MIN(InvoiceDate), MAX(InvoiceDate) FROM transactions")
    min_date, max_date = cursor.fetchone()
    print(f"✅ Date range: {min_date} to {max_date}")
    
    print("\n📋 Sample data (first 5 rows):")
    cursor.execute("SELECT TOP 5 * FROM transactions")
    for row in cursor.fetchall():
        print(f"   {row}")

except Exception as e:
    print(f"❌ Error during verification: {e}")


📊 STEP 6: Verifying data in SQL...
❌ Error during verification: The cursor's connection has been closed.


In [ ]:
# ============ CLOSE CONNECTION ============
try:
    conn.close()
    print("\n✅ Connection closed successfully")
except Exception as e:
    print(f"⚠️  Error closing connection: {e}")

print("\n" + "=" * 60)
print("✅ DATA LOADING COMPLETE!!")
print("=" * 60)
print("\nNext steps:")
print("1. Open Power BI Desktop")
print("2. Get Data → SQL Server Database")
print("3. Server: (localdb)\\MSSQLLocalDB")
print("4. Database: RevOpsAnalytics")
print("5. Select 'transactions' table → Load")
print("6. Create your RevOps dashboards!")
print("\n" + "=" * 60)

⚠️  Error closing connection: Attempt to use a closed connection.

✅ DATA LOADING COMPLETE!

Next steps:
1. Open Power BI Desktop
2. Get Data → SQL Server Database
3. Server: (localdb)\MSSQLLocalDB
4. Database: RevOpsAnalytics
5. Select 'transactions' table → Load
6. Create your RevOps dashboards!

